In [1]:
# =========================================================
# DATAHUB SETUP & IMPORTS. (May take a while!) Comment me out after first run!
# =========================================================

# remove old kernel
!jupyter kernelspec uninstall -y cse151b

# Remove old .venv
!rm -rf .venv

print("Old environment and kernel destroyed.")

# Install uv
!wget -qO- https://astral.sh/uv/install.sh | sh

# Create a virtual environment
!~/.local/bin/uv venv .venv --seed

# Install dependencies — this is fast thanks to uv's parallel resolver
!.venv/bin/python -m pip install sympy numpy transformers dspy vllm==0.19.1 tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# Install Jupyter Kernel
!.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

print("Done. Restart the kernel before proceeding. Then click on current kernel -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'. Restart kernel again. Comment out this code!")

Couldn't find kernel spec(s): cse151b


rm: cannot remove '.venv/lib/python3.13/site-packages/tornado/.nfs000000000140417300000c9e': Device or resource busy
rm: cannot remove '.venv/lib/python3.13/site-packages/psutil/.nfs000000000140601500000c9f': Device or resource busy
rm: cannot remove '.venv/lib/python3.13/site-packages/pyzmq.libs/.nfs000000000140543900000ca0': Device or resource busy
rm: cannot remove '.venv/lib/python3.13/site-packages/pyzmq.libs/.nfs000000000140543800000ca1': Device or resource busy
rm: cannot remove '.venv/lib/python3.13/site-packages/zmq/backend/cython/.nfs000000000140546100000ca2': Device or resource busy
rm: cannot remove '.venv/lib/python3.13/site-packages/debugpy/_vendored/pydevd/_pydevd_sys_monitoring/.nfs00000000009a9f0d00000ca3': Device or resource busy
rm: cannot remove '.venv/lib/python3.13/site-packages/debugpy/_vendored/pydevd/_pydevd_bundle/.nfs00000000009a9eb100000ca4': Device or resource busy


downloading uv 0.11.17 x86_64-unknown-linux-gnu


installing to /home/a5verma/.local/bin


  uv
  uvx
everything's installed!


...

Old environment and kernel destroyed.


Using CPython 3.13.13 interpreter at: /opt/conda/bin/python
Creating virtual environment with seed packages at: .venv
error: Failed to create virtual environment
  Caused by: A directory already exists at: .venv

hint: Use the `--clear` flag or set `UV_VENV_CLEAR=1` to replace the existing directory


/bin/bash: line 1: .venv/bin/python: No such file or directory


/bin/bash: line 1: .venv/bin/python: No such file or directory


Done. Restart the kernel before proceeding. Then click on current kernel -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'. Restart kernel again. Comment out this code!


In [5]:
import dspy
from dspy.teleprompt import GEPA
import json
import os
import re
import sys
from pathlib import Path
from typing import Optional

In [ ]:
# 1. Define your module (This must be identical in both files)
class TextClassification(dspy.Signature):
    """Classify the sentiment of the provided text."""
    text = dspy.InputField()
    sentiment = dspy.OutputField(desc="positive, negative, or neutral")

class CoTClassifier(dspy.Module):
    def __init__(self):
        super().__init__()
        self.prog = dspy.ChainOfThought(TextClassification)
        
    def forward(self, text):
        return self.prog(text=text)

def exact_match_metric(example, pred, trace=None):
    return example.sentiment.strip().lower() == pred.sentiment.strip().lower()

# 2. Configure the LM used specifically for optimization
# You might use a larger model here to act as a better "proposer" for GEPA
dspy.settings.configure(lm=dspy.LM('openai/gpt-4o'))

# 3. Load data and compile
trainset = [...] # Your training data here
optimizer = GEPA(metric=exact_match_metric)

# Run the evolutionary search
optimized_classifier = optimizer.compile(
    student=CoTClassifier(), 
    trainset=trainset
)

# 4. Save the optimized state to a separate file
# This creates a JSON file containing the evolved prompts and examples.
optimized_classifier.save("optimized_prompts.json")
print("Optimization complete. Prompts saved to JSON.")

In [ ]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices carefully. "
    "Think through the problem step by step, then select the single best answer. "
    "At the very end of your response, output the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)

def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question

In [6]:
STRATIFIED_DATA   = "data/16k_max_openr1_math_7_5k_stratified.jsonl"
PUBLIC_DATA       = "data/public.jsonl"

data = [json.loads(line) for line in open(STRATIFIED_DATA)]

In [ ]:
# "Shared Definitions"

import dspy

# 1. Define the Math Architecture
class MathReasoning(dspy.Signature):
    """Solve the math problem step-by-step."""
    question = dspy.InputField()
    reasoning = dspy.OutputField(desc="Step-by-step logical deductions")
    answer = dspy.OutputField(desc="The final numerical answer only")

class MathSolver(dspy.Module):
    def __init__(self):
        super().__init__()
        self.prog = dspy.ChainOfThought(MathReasoning)
        
    def forward(self, question):
        return self.prog(question=question)

# 2. Define the Metric
# Returning a dspy.Prediction object allows GEPA to read the 'feedback' 
# while allowing dspy.Evaluate to read and aggregate the numeric 'score'.
def math_feedback_metric(example, pred, trace=None):
    correct = str(example.answer).strip().lower() == str(pred.answer).strip().lower()
    score = 1.0 if correct else 0.0
    
    feedback = "Correctly solved." if correct else (
        f"Incorrect. Expected: {example.answer}, Got: {pred.answer}.\n"
        f"The model's trace was: {pred.reasoning}\n"
        f"Identify the logical flaw and update instructions to prevent it."
    )
    return dspy.Prediction(score=score, feedback=feedback)

In [ ]:
# Prompt optimization

from dspy.teleprompt import GEPA

# 1. Load only the training data
trainset, _ = load_math_data()

# 2. Configure the large reflection model to run the optimizer
dspy.settings.configure(lm=dspy.LM('openai/gpt-4o'))

# 3. Compile the prompt using GEPA
optimizer = GEPA(metric=math_feedback_metric)
optimized_solver = optimizer.compile(
    student=MathSolver(),
    trainset=trainset
)

# 4. Export the optimized instructions and few-shot traces
optimized_solver.save("math_prompts.json")
print("Optimization complete. Prompts saved to math_prompts.json.")

In [ ]:
# Evaluation

from dspy.evaluate import Evaluate

# 1. Load only the unseen test set (ensuring zero data leakage)
_, testset = load_math_data()

# 2. Configure Qwen3-4B-Thinking-2507 using Hugging Face
# We apply the official sampling parameters recommended by Alibaba for the 2507 Thinking series.
qwen_model = dspy.LM(
    "huggingface/Qwen/Qwen3-4B-Thinking-2507",
    temperature=0.6,      # Lower temperature is critical for strict reasoning coherence
    top_p=0.95,          # Allows a wider pool of precise mathematical tokens
    top_k=20,            # Filters out highly unlikely tokens early
    min_p=0.0,
    max_tokens=16384,    # Mandated large token limit so the model doesn't truncate mid-thought
    device_map="auto"    # Automatically sharded across available GPUs
)
dspy.settings.configure(lm=qwen_model)

# 3. Instantiate your pipeline and load the parameters generated by GEPA
production_solver = MathSolver()
production_solver.load("math_prompts.json")

# 4. Execute the evaluation over your unseen data
evaluator = Evaluate(
    devset=testset, 
    metric=math_feedback_metric, 
    num_threads=2,        # Tweak based on your active GPU VRAM/Concurrency limits
    display_progress=True,
    display_table=True
)

eval_result = evaluator(production_solver)
print(f"\nFinal Accuracy of Qwen3-4B-Thinking-2507 on Unseen Test Set: {eval_result.score * 100:.2%}")